In [ ]:
import cv2
import numpy as np

# Carrega  a imagem
img = cv2.imread("dog.jpg") # carrega a imagem com OpenCV

# mapeia o tamanho da imagem para alocar mémoria para a matriz e percorrer os pixels nos loops
h, w, _ = img.shape # pega as dimensões da imagens (linhas e colunas da matriz, ignorando a terceira dimensão da imagem, que são os 3 canais de cor)

# Transforma em escala de cinza
gray = np.zeros((h, w), dtype=np.float32) # cria a matriz vazia

# percorre os pixels
for y in range(h):
    for x in range(w):
      # extrai individualmente o valor de cada componente R, G e B do pixel
        b = img[y, x, 0]
        g = img[y, x, 1]
        r = img[y, x, 2]

        # média ponderada das componentes para conversão em cinza pela fórmula da luminosidade
        gray[y, x] = 0.114*b + 0.587*g + 0.299*r # armazena a média na matriz

# Aplica Convolução, suavizando a imagem com média ponderada local
def convolucao(img, kernel):

    kh, kw = kernel.shape

    ph = kh // 2
    pw = kw // 2

    out = np.zeros_like(img)

    padded = np.pad(
        img,
        ((ph, ph), (pw, pw)),
        mode='edge'
    )

    for y in range(img.shape[0]):
        for x in range(img.shape[1]):

            soma = 0

            for ky in range(kh):
                for kx in range(kw):

                    soma += (
                        padded[y+ky, x+kx]
                        * kernel[ky, kx]
                    )

            out[y, x] = soma

    return out

# Aplica Blur com Filtro Gausiano 6x para suavizar e eliminar ruído
gauss = np.array([
    [1,2,1],
    [2,4,2],
    [1,2,1]
], dtype=np.float32)

gauss /= 16.0 # Normaliza para que a soma do kernel seja igual a 1

blur = gray.copy()

for _ in range(6):
    blur = convolucao(blur, gauss)


# Detecta fundo claro
# Separa a imagem drasticamente em branco e preto para separar o fundo mais claro do cachorro mais escuro.
binary = np.zeros((h, w), dtype=np.uint8)

limiar = 170 # limite da luminosidade

for y in range(h):
    for x in range(w):

        # fundo branco
        if blur[y, x] > limiar:
            binary[y, x] = 255 # converte para branco puro
        else:
            binary[y, x] = 0 # converte para preto puro

# Flood Fill
# Para separar evidentemente o interior do cachorro e o exterior da imagem,
# pois, no meu projeto, não quero que a tartaruga desenhe detalhes interno do cachorro,
# como olhos e nariz, mas sim que contorne a sua borda.

visited = np.zeros((h, w), dtype=bool)

stack = [(0, 0)] # Inicializa uma pilha :)

visited[0,0] = True

direcoes = [
    (-1,0),
    (1,0),
    (0,-1),
    (0,1)
]

while stack:

    y, x = stack.pop()

    for dy, dx in direcoes:

        ny = y + dy
        nx = x + dx

        if (
            0 <= ny < h and
            0 <= nx < w
        ):

            if (
                binary[ny, nx] == 255 and
                not visited[ny, nx]
            ):

                visited[ny, nx] = True
                stack.append((ny, nx))


# Inverte
# Lógica que pensei: se a coordenada não foi visitada pelo loop supracitado,
# significa que ela faz parte do cachorro (seja pelo fato de ser escura originalmente ou por ser uma mancha interna isolada).

dog = np.zeros((h, w), dtype=np.uint8)

for y in range(h):
    for x in range(w):

        # tudo que não é fundo
        if not visited[y, x]:
            dog[y, x] = 255


# Extrai o Contorno Externo

edge = np.zeros((h, w), dtype=np.uint8)

for y in range(1, h-1):
    for x in range(1, w-1):

        if dog[y, x] == 255: # Verifica se o pixel atual pertence à silhueta do cachorro.

            # Coleta os valores de intensidade dos vizinhos
            vizinhos = [
                dog[y-1, x],
                dog[y+1, x],
                dog[y, x-1],
                dog[y, x+1]
            ]

            # Se o pixel é branco, mas pelo menos um de seus vizinhos é preto (0),
            # significa que esse pixel está exatamente na fronteira entre o cachorro e o fundo. Logo, ele é marcado como borda.
            if 0 in vizinhos:
                edge[y, x] = 255

# Engrossa o Contorno, pois o mapeamento de pontos não estava tão preciso para o caminho da tartaruga
# apenas com o contorno de raio padrão igual a 1.
thick = np.zeros((h, w), dtype=np.uint8)

raio = 2

for y in range(h):
    for x in range(w):

        if edge[y, x] == 255:

            # pinta vizinhos ao redor
            for dy in range(-raio, raio + 1):
                for dx in range(-raio, raio + 1):

                    ny = y + dy
                    nx = x + dx

                    if (
                        0 <= ny < h and
                        0 <= nx < w
                    ):

                        thick[ny, nx] = 255

# gera imagem com borda
cv2.imwrite("contorno.png", thick)